# Deep Past Challenge — Akkadian → English Translation

This notebook loads trained models and generates English translations from Akkadian transliterations.

**Setup:** Add the relevant model dataset(s) to this notebook:
- ByT5: `byt5-akkadian-checkpoint`
- BiLSTM: `bilstm-akkadian-checkpoint`
- Transformer: `transformer-akkadian-checkpoint`

**Choose your model** in the next cell.

In [ ]:
ACTIVE_MODEL = "byt5"  # Options: "byt5", "bilstm", "transformer"
BYTA_VARIANT = "sft"   # ByT5 variant: "base" or "sft" (only used if ACTIVE_MODEL == "byt5")

# ── Shared imports & device setup ───────────────────────────────────────
import os
import re
import math
import glob
import unicodedata
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import (pack_padded_sequence, pad_packed_sequence,
                                pad_sequence)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"Active model: {ACTIVE_MODEL}")
if ACTIVE_MODEL == "byt5":
    print(f"ByT5 variant: {BYTA_VARIANT}")

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────
OUTPUT_CSV = '/kaggle/working/submission.csv'
INPUT_ROOT = '/kaggle/input'

NUM_BEAMS = 4
MAX_LENGTH = 512
BATCH_SIZE = 16

# Auto-detect test.csv path
test_matches = glob.glob(os.path.join(INPUT_ROOT, '**', 'test.csv'), recursive=True)
if not test_matches:
    raise FileNotFoundError(f"No test.csv found under {INPUT_ROOT}")
TEST_CSV = test_matches[0]
print(f"Test CSV: {TEST_CSV}")

In [ ]:
# ── Preprocessing ──────────────────────────────────────────────────────
_SUBSCRIPT_MAP = str.maketrans('\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089', '0123456789')
_GLOTTAL_CHARS = '\u02BE\u02BF\u02BC\u02BB\u0027\u2018\u2019'  # ʾ ʿ ʼ ʻ ' ' '

# Determinative normalization patterns (critical for Akkadian)
_DETERMINATIVE_PATTERNS = [
    (re.compile(r'\{d\}', re.IGNORECASE), '{d}'),
    (re.compile(r'\bDINGIR\b'), '{d}'),
    (re.compile(r'\{ki\}', re.IGNORECASE), '{ki}'),
    (re.compile(r'\{m\}', re.IGNORECASE), '{m}'),
    (re.compile(r'\{f\}', re.IGNORECASE), '{f}'),
    (re.compile(r'\{mi\}', re.IGNORECASE), '{f}'),
    (re.compile(r'\{munus\}', re.IGNORECASE), '{f}'),
    (re.compile(r'\{mesh\}', re.IGNORECASE), '{mesh}'),
    (re.compile(r'\{meš\}', re.IGNORECASE), '{mesh}'),
    (re.compile(r'\{gish\}', re.IGNORECASE), '{gish}'),
    (re.compile(r'\{giš\}', re.IGNORECASE), '{gish}'),
    (re.compile(r'\{ĝeš\}', re.IGNORECASE), '{gish}'),
    (re.compile(r'\{geš\}', re.IGNORECASE), '{gish}'),
    (re.compile(r'\{gi\}', re.IGNORECASE), '{gi}'),
    (re.compile(r'\{na4\}', re.IGNORECASE), '{na4}'),
    (re.compile(r'\{na₄\}', re.IGNORECASE), '{na4}'),
    (re.compile(r'\{urudu\}', re.IGNORECASE), '{urudu}'),
    (re.compile(r'\{tug2\}', re.IGNORECASE), '{tug2}'),
    (re.compile(r'\{tug₂\}', re.IGNORECASE), '{tug2}'),
    (re.compile(r'\{túg\}', re.IGNORECASE), '{tug2}'),
    (re.compile(r'\{uru\}', re.IGNORECASE), '{uru}'),
    (re.compile(r'\{kur\}', re.IGNORECASE), '{kur}'),
    (re.compile(r'\{mul\}', re.IGNORECASE), '{mul}'),
    (re.compile(r'\{e₂\}', re.IGNORECASE), '{e2}'),
    (re.compile(r'\{é\}', re.IGNORECASE), '{e2}'),
    (re.compile(r'\{dub\}', re.IGNORECASE), '{dub}'),
    (re.compile(r'\{id₂\}', re.IGNORECASE), '{id2}'),
    (re.compile(r'\{íd\}', re.IGNORECASE), '{id2}'),
    (re.compile(r'\{mušen\}', re.IGNORECASE), '{mushen}'),
    (re.compile(r'\{kuš\}', re.IGNORECASE), '{kush}'),
    (re.compile(r'\{u₂\}', re.IGNORECASE), '{u2}'),
    (re.compile(r'\{ú\}', re.IGNORECASE), '{u2}'),
    (re.compile(r'\{lu₂\}', re.IGNORECASE), '{lu2}'),
    (re.compile(r'\{lú\}', re.IGNORECASE), '{lu2}'),
]

_VOWEL_ACCENTS = [
    (re.compile(r'a2\b'), '\u00e1'), (re.compile(r'a3\b'), '\u00e0'),
    (re.compile(r'e2\b'), '\u00e9'), (re.compile(r'e3\b'), '\u00e8'),
    (re.compile(r'i2\b'), '\u00ed'), (re.compile(r'i3\b'), '\u00ec'),
    (re.compile(r'u2\b'), '\u00fa'), (re.compile(r'u3\b'), '\u00f9'),
    (re.compile(r'A2\b'), '\u00c1'), (re.compile(r'A3\b'), '\u00c0'),
    (re.compile(r'E2\b'), '\u00c9'), (re.compile(r'E3\b'), '\u00c8'),
    (re.compile(r'I2\b'), '\u00cd'), (re.compile(r'I3\b'), '\u00cc'),
    (re.compile(r'U2\b'), '\u00da'), (re.compile(r'U3\b'), '\u00d9'),
]

def clean_transliteration(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\u1e2a', 'H').replace('\u1e2b', 'h')
    # Remove glottal characters (ʾ ʿ ʼ ʻ ' ' ')
    for ch in _GLOTTAL_CHARS:
        text = text.replace(ch, '')
    text = text.translate(_SUBSCRIPT_MAP)
    # Normalize determinatives (critical for Akkadian linguistic markers)
    for pattern, replacement in _DETERMINATIVE_PATTERNS:
        text = pattern.sub(replacement, text)
    # Gap markers
    text = re.sub(r'\{large break\}', '<big_gap>', text, flags=re.IGNORECASE)
    text = re.sub(r'\{big gap\}', '<big_gap>', text, flags=re.IGNORECASE)
    text = re.sub(r'\[…\s*…\]', '<big_gap>', text)
    text = re.sub(r'…', '<big_gap>', text)
    text = re.sub(r'\[(?:x\s*)+\]', '<gap>', text)
    text = re.sub(r'\[\.{2,}\]', '<gap>', text)
    text = re.sub(r'\.{3,}', '<gap>', text)
    text = re.sub(r'\[broken\]', '<gap>', text, flags=re.IGNORECASE)
    text = re.sub(r'\[damaged\]', '<gap>', text, flags=re.IGNORECASE)
    text = re.sub(r'\[missing\]', '<gap>', text, flags=re.IGNORECASE)
    # Half-brackets and bracket cleanup
    text = re.sub(r'[\u02f9\u2e22\u02fa\u2e23]', '', text)
    text = re.sub(r'<<[^>]*>>', '', text)
    text = re.sub(r'\[([^\]]*)\]', r'\1', text)
    text = re.sub(r'<(?!gap>|big_gap>)([^>]*)>', r'\1', text)
    text = re.sub(r'[!?*#]', '', text)
    # Romanization variants
    text = text.replace('sz', '\u0161').replace('SZ', '\u0160')
    text = re.sub(r's,', '\u1e63', text)
    text = re.sub(r'S,', '\u1e62', text)
    text = re.sub(r't,', '\u1e6d', text)
    text = re.sub(r'T,', '\u1e6c', text)
    # Accented vowels
    for p, r in _VOWEL_ACCENTS:
        text = p.sub(r, text)
    # Deduplicate gaps
    text = re.sub(r'(<big_gap>\s*){2,}', '<big_gap> ', text)
    text = re.sub(r'(<gap>\s*){2,}', '<gap> ', text)
    text = re.sub(r'(<(?:big_)?gap>\s*){2,}', '<big_gap> ', text)
    # Whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Quick test
print(clean_transliteration('um-ma kà-ru-um kà-ni-ia-ma'))


In [ ]:
# ── Load and Prepare Test Data ──────────────────────────────────────────
test_df = pd.read_csv(TEST_CSV)
print(f'Test samples: {len(test_df)}')
print(f'Columns: {list(test_df.columns)}')
print(test_df.head())

translit_col = next((c for c in test_df.columns if 'translit' in c.lower()), test_df.columns[-1])
id_col = 'id' if 'id' in test_df.columns else test_df.columns[0]
print(f'\nUsing transliteration column: {translit_col}')
print(f'Using ID column: {id_col}')

test_df['clean'] = test_df[translit_col].apply(clean_transliteration)
print(f'\nSample cleaned:')
print(test_df[['clean']].head())

In [ ]:
## ByT5-base (Pre-trained Encoder-Decoder)

In [ ]:
# ── ByT5 — Load & Infer ─────────────────────────────────────────────────
byt5_predictions = None

if ACTIVE_MODEL == "byt5":
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    # Auto-detect model path (config.json) with variant filtering
    matches = glob.glob(os.path.join(INPUT_ROOT, '**', 'config.json'), recursive=True)
    if not matches:
        raise FileNotFoundError(f"No config.json found under {INPUT_ROOT}")
    
    # Filter by variant if multiple ByT5 datasets attached
    if BYTA_VARIANT == "sft":
        sft_matches = [m for m in matches if 'sft' in m.lower()]
        MODEL_PATH = os.path.dirname(sft_matches[0]) if sft_matches else os.path.dirname(matches[0])
    else:  # base variant
        base_matches = [m for m in matches if 'sft' not in m.lower()]
        MODEL_PATH = os.path.dirname(base_matches[0]) if base_matches else os.path.dirname(matches[0])
    
    print(f"ByT5 ({BYTA_VARIANT}) model path: {MODEL_PATH}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    byt5_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH, local_files_only=True).to(DEVICE)
    byt5_model.eval()
    print("ByT5 model loaded.")

    texts = test_df['clean'].tolist()
    byt5_predictions = []

    for i in range(0, len(texts), BATCH_SIZE):
        batch = [t if t else '<gap>' for t in texts[i:i + BATCH_SIZE]]
        inputs = tokenizer(
            batch, max_length=MAX_LENGTH, truncation=True,
            padding=True, return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            outputs = byt5_model.generate(
                **inputs, num_beams=NUM_BEAMS,
                max_length=MAX_LENGTH, early_stopping=True)

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        byt5_predictions.extend(decoded)

        done = min(i + BATCH_SIZE, len(texts))
        if done % 50 == 0 or done == len(texts):
            print(f'  ByT5: {done}/{len(texts)}')

    print(f'ByT5 predictions: {len(byt5_predictions)}')
else:
    print("ByT5 skipped (ACTIVE_MODEL != 'byt5').")

In [ ]:
## BiLSTM Seq2Seq with Bahdanau Attention

In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# ── Character-level vocabulary ──────────────────────────────────────────
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

class CharVocab:
    def __init__(self):
        self.char2idx = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.idx2char = {v: k for k, v in self.char2idx.items()}
        self.size = 4

    def build_from_dict(self, char2idx):
        self.char2idx = char2idx
        self.idx2char = {v: k for k, v in char2idx.items()}
        self.size = len(char2idx)

    def encode(self, text, max_len=300):
        ids = [SOS_IDX]
        for c in text[:max_len - 2]:
            ids.append(self.char2idx.get(c, UNK_IDX))
        ids.append(EOS_IDX)
        return ids

    def decode(self, ids):
        chars = []
        for idx in ids:
            if idx == EOS_IDX:
                break
            if idx not in (PAD_IDX, SOS_IDX):
                chars.append(self.idx2char.get(idx, "?"))
        return "".join(chars)


# ── BiLSTM Seq2Seq Architecture ─────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                           bidirectional=True, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0)
        self.fc_h = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_c = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_lens):
        embedded = self.dropout(self.embedding(src))
        packed = pack_padded_sequence(embedded, src_lens.cpu().clamp(min=1),
                                     batch_first=True, enforce_sorted=False)
        outputs, (hidden, cell) = self.rnn(packed)
        outputs, _ = pad_packed_sequence(outputs, batch_first=True)
        hidden = torch.cat([hidden[0::2], hidden[1::2]], dim=2)
        cell = torch.cat([cell[0::2], cell[1::2]], dim=2)
        hidden = torch.tanh(self.fc_h(hidden))
        cell = torch.tanh(self.fc_c(cell))
        return outputs, hidden, cell


class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim, enc_dim):
        super().__init__()
        self.W_q = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_k = nn.Linear(enc_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, mask):
        query = self.W_q(decoder_hidden).unsqueeze(1)
        keys = self.W_k(encoder_outputs)
        energy = self.v(torch.tanh(query + keys)).squeeze(2)
        energy = energy.masked_fill(mask == 0, -1e10)
        attn_weights = torch.softmax(energy, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        return context.squeeze(1), attn_weights


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, enc_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.attention = BahdanauAttention(hidden_dim, enc_dim)
        self.rnn = nn.LSTM(embed_dim + enc_dim, hidden_dim, num_layers=num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc_out = nn.Linear(hidden_dim + enc_dim + embed_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, cell, encoder_outputs, mask):
        embedded = self.dropout(self.embedding(input_token.unsqueeze(1)))
        context, attn_weights = self.attention(hidden[-1], encoder_outputs, mask)
        rnn_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        prediction = self.fc_out(
            torch.cat([output.squeeze(1), context, embedded.squeeze(1)], dim=1))
        return prediction, hidden, cell, attn_weights


class BiLSTMSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def translate(self, src, src_lens, max_len=300):
        self.eval()
        with torch.no_grad():
            encoder_outputs, hidden, cell = self.encoder(src, src_lens)
            mask = (src != PAD_IDX).float()
            batch_size = src.size(0)
            input_token = torch.full((batch_size,), SOS_IDX, dtype=torch.long, device=self.device)
            all_tokens = []
            for _ in range(max_len):
                prediction, hidden, cell, _ = self.decoder(
                    input_token, hidden, cell, encoder_outputs, mask)
                input_token = prediction.argmax(1)
                all_tokens.append(input_token.unsqueeze(1))
                if (input_token == EOS_IDX).all():
                    break
            return torch.cat(all_tokens, dim=1)

print("BiLSTM model classes defined.")

In [ ]:
# ── BiLSTM — Load & Infer ───────────────────────────────────────────────
bilstm_predictions = None

if ACTIVE_MODEL == "bilstm":
    bilstm_matches = glob.glob(os.path.join(INPUT_ROOT, '**', 'best_model.pt'), recursive=True)
    bilstm_path = [p for p in bilstm_matches if 'bilstm' in p.lower()]
    if not bilstm_path:
        bilstm_path = bilstm_matches
    BILSTM_CKPT = bilstm_path[0] if bilstm_path else None

    if not BILSTM_CKPT:
        raise FileNotFoundError("BiLSTM best_model.pt not found under /kaggle/input/")

    print(f"BiLSTM checkpoint: {BILSTM_CKPT}")
    ckpt = torch.load(BILSTM_CKPT, map_location=DEVICE, weights_only=False)

    src_vocab_bilstm = CharVocab()
    tgt_vocab_bilstm = CharVocab()
    src_vocab_bilstm.build_from_dict(ckpt["src_vocab"])
    tgt_vocab_bilstm.build_from_dict(ckpt["tgt_vocab"])

    enc = Encoder(src_vocab_bilstm.size, 256, 512, 2, 0.3)
    dec = Decoder(tgt_vocab_bilstm.size, 256, 512, 512 * 2, 2, 0.3)
    bilstm_model = BiLSTMSeq2Seq(enc, dec, DEVICE).to(DEVICE)
    bilstm_model.load_state_dict(ckpt["model_state_dict"])
    bilstm_model.eval()
    print(f"BiLSTM loaded — epoch {ckpt.get('epoch', '?')}, geo_mean {ckpt.get('geo_mean', '?'):.2f}")

    texts_clean = test_df['clean'].tolist()
    bilstm_predictions = []

    for i in range(0, len(texts_clean), 64):
        batch_texts = texts_clean[i : i + 64]
        encoded = [torch.tensor(src_vocab_bilstm.encode(t if t else "<gap>"), dtype=torch.long)
                   for t in batch_texts]
        src_lens = torch.tensor([len(e) for e in encoded])
        src_padded = pad_sequence(encoded, batch_first=True, padding_value=PAD_IDX).to(DEVICE)

        output_tokens = bilstm_model.translate(src_padded, src_lens)
        for row in output_tokens.cpu().numpy():
            bilstm_predictions.append(tgt_vocab_bilstm.decode(row))

        done = min(i + 64, len(texts_clean))
        if done % 128 == 0 or done == len(texts_clean):
            print(f"  BiLSTM: {done}/{len(texts_clean)}")

    print(f"BiLSTM predictions: {len(bilstm_predictions)}")
else:
    print("BiLSTM skipped (ACTIVE_MODEL != 'bilstm').")

## Vanilla Transformer (from scratch)

In [ ]:
import math

# ── Transformer Architecture ────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class TransformerMT(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, n_heads,
                 n_layers, d_ff, dropout, max_len):
        super().__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_IDX)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=n_heads,
            num_encoder_layers=n_layers, num_decoder_layers=n_layers,
            dim_feedforward=d_ff, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def translate(self, src, max_len=300):
        self.eval()
        with torch.no_grad():
            src_pad_mask = (src == PAD_IDX)
            src_emb = self.pos_enc(self.src_embedding(src) * math.sqrt(self.d_model))
            memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_pad_mask)
            batch_size = src.size(0)
            ys = torch.full((batch_size, 1), SOS_IDX, dtype=torch.long, device=src.device)
            for _ in range(max_len):
                tgt_emb = self.pos_enc(self.tgt_embedding(ys) * math.sqrt(self.d_model))
                tgt_mask = self.transformer.generate_square_subsequent_mask(
                    ys.size(1), device=src.device)
                output = self.transformer.decoder(
                    tgt_emb, memory, tgt_mask=tgt_mask,
                    memory_key_padding_mask=src_pad_mask)
                logits = self.fc_out(output[:, -1:])
                next_token = logits.argmax(-1)
                ys = torch.cat([ys, next_token], dim=1)
                if (next_token == EOS_IDX).all():
                    break
            return ys[:, 1:]

print("Transformer model classes defined.")

In [ ]:
# ── Transformer — Load & Infer ──────────────────────────────────────────
tf_predictions = None

if ACTIVE_MODEL == "transformer":
    transformer_matches = glob.glob(os.path.join(INPUT_ROOT, '**', 'best_model.pt'), recursive=True)
    transformer_path = [p for p in transformer_matches if 'transformer' in p.lower()]
    if not transformer_path:
        transformer_path = transformer_matches
    TRANSFORMER_CKPT = transformer_path[0] if transformer_path else None

    if not TRANSFORMER_CKPT:
        raise FileNotFoundError("Transformer best_model.pt not found under /kaggle/input/")

    print(f"Transformer checkpoint: {TRANSFORMER_CKPT}")
    t_ckpt = torch.load(TRANSFORMER_CKPT, map_location=DEVICE, weights_only=False)

    src_vocab_tf = CharVocab()
    tgt_vocab_tf = CharVocab()
    src_vocab_tf.build_from_dict(t_ckpt["src_vocab"])
    tgt_vocab_tf.build_from_dict(t_ckpt["tgt_vocab"])

    tf_model = TransformerMT(
        src_vocab_tf.size, tgt_vocab_tf.size,
        d_model=256, n_heads=4, n_layers=4, d_ff=512,
        dropout=0.1, max_len=300
    ).to(DEVICE)
    tf_model.load_state_dict(t_ckpt["model_state_dict"])
    tf_model.eval()
    print(f"Transformer loaded — epoch {t_ckpt.get('epoch', '?')}, geo_mean {t_ckpt.get('geo_mean', '?'):.2f}")

    texts_clean = test_df['clean'].tolist()
    tf_predictions = []

    for i in range(0, len(texts_clean), 64):
        batch_texts = texts_clean[i : i + 64]
        encoded = [torch.tensor(src_vocab_tf.encode(t if t else "<gap>"), dtype=torch.long)
                   for t in batch_texts]
        src_padded = pad_sequence(encoded, batch_first=True, padding_value=PAD_IDX).to(DEVICE)

        output_tokens = tf_model.translate(src_padded)
        for row in output_tokens.cpu().numpy():
            tf_predictions.append(tgt_vocab_tf.decode(row))

        done = min(i + 64, len(texts_clean))
        if done % 128 == 0 or done == len(texts_clean):
            print(f"  Transformer: {done}/{len(texts_clean)}")

    print(f"Transformer predictions: {len(tf_predictions)}")
else:
    print("Transformer skipped (ACTIVE_MODEL != 'transformer').")

In [ ]:
# ── Create Submission from Active Model ──────────────────────────────────
model_predictions = {
    "byt5": byt5_predictions,
    "bilstm": bilstm_predictions,
    "transformer": tf_predictions,
}

chosen = model_predictions[ACTIVE_MODEL]
if chosen is None:
    raise RuntimeError(f"No predictions for ACTIVE_MODEL='{ACTIVE_MODEL}'. Check checkpoint.")

submission = pd.DataFrame({
    'id': test_df[id_col],
    'translation': chosen,
})
submission['translation'] = submission['translation'].fillna('')
submission.to_csv(OUTPUT_CSV, index=False)

print(f'Model: {ACTIVE_MODEL}')
print(f'Submission saved to: {OUTPUT_CSV}')
print(f'Shape: {submission.shape}')
print(f'\nFirst 5 predictions:')
for _, row in submission.head(5).iterrows():
    print(f"  [{row['id']}]: {row['translation'][:120]}")

# Verify format against sample
sample_matches = glob.glob(os.path.join(INPUT_ROOT, '**', 'sample_submission.csv'), recursive=True)
sample_path = sample_matches[0] if sample_matches else None
if sample_path and os.path.exists(sample_path):
    sample = pd.read_csv(sample_path)
    print(f'\nFormat check:')
    print(f'  Expected columns: {list(sample.columns)}')
    print(f'  Got columns:      {list(submission.columns)}')
    print(f'  Expected rows:    {len(sample)}')
    print(f'  Got rows:         {len(submission)}')
    match = list(sample.columns) == list(submission.columns)
    print(f'  Columns match: {"YES" if match else "NO"}')